# 예제 franka_ex13: FR3 YZ 평면 원형 경로 (Cartesian Path)

끝단을 YZ 평면에서 원을 3 회 반복 그리도록 한다 — 오프라인으로 전체 궤적을 미리 계획하는 방식.

**6-DOF 예제와 다른 점**
- 원 반경 키움: 10cm → **15cm** (FR3 reach 가 큼)
- 원의 중심 위치: 6-DOF `(0.35, 0.10, 0.40)` → FR3 `(0.55, 0.20, 0.55)` — base 로부터 멀고 위에 있는 위치
- 끝단 자세: `pitch=+π/2` 로 그리퍼가 +X 방향을 향한다 (회전축이 X)
- 끝단 링크 `fr3_hand_tcp`, planning group `fr3_arm`
- 7-DOF redundancy 덕에 같은 원이 6-DOF 보다 잘 풀린다 — 보통 fraction=1.0
- `home` 없음 → 시작/복귀는 `ready`

**학습 내용**
- 원 위 균등 분할 waypoints (24점/회 × 3회 + 1)
- `compute_cartesian_path(max_step=0.02)` — 2cm 직선 보간
- `fraction` 의 의미 (얼마나 풀렸는지)
- `ExecuteTrajectory` 액션으로 미리 계산된 trajectory 만 재생
- RViz LINE_STRIP 으로 목표 / 실제 경로 비교

## 실행 절차

이 노트북은 별도로 띄운 MoveIt + RViz 의 `move_group` 액션 서버에 클라이언트로 붙는다.

> ⚠ 다른 로봇용 MoveIt launch 가 떠 있으면 같은 토픽으로 충돌할 수 있다.
> 시작 전에 `pgrep -af 'ros2 launch'` 로 잔존 프로세스가 없는지 확인하자.

### 터미널 1 — Franka FR3 (Gazebo Sim) + MoveIt + RViz 기동

```bash
source /opt/ros/jazzy/setup.bash
source ~/robot_arm/install/setup.bash
ros2 launch franka_tutorials franka_gazebo_moveit.launch.py
```

RViz 가 뜨면 **`MarkerArray` Display 를 추가하고 Topic 을 `/path_markers` 로 설정**한다.
Fixed Frame 은 `fr3_link0` 로 둔다.

### 터미널 2 — Jupyter 기동

```bash
source ~/venv/ros_jazzy/bin/activate
source /opt/ros/jazzy/setup.bash
source ~/robot_arm/install/setup.bash
cd ~/robot_arm/src/robotarm_tutorials/robot_arm_tutorials/robot_arm_tutorials
jupyter lab
```

셀을 위에서 아래로 순서대로 실행한다 (`Shift+Enter`).

## 1. 상수

In [ ]:
PLANNING_GROUP    = 'fr3_arm'
REFERENCE_FRAME   = 'fr3_link0'
END_EFFECTOR_LINK = 'fr3_hand_tcp'
ARM_JOINTS        = ['fr3_joint1', 'fr3_joint2', 'fr3_joint3',
                     'fr3_joint4', 'fr3_joint5', 'fr3_joint6',
                     'fr3_joint7']
MARKER_TOPIC = '/path_markers'

RADIUS     = 0.15        # 원 반경 (FR3 스케일)
PTS_PER_LOOP = 24
NUM_LOOPS    = 3
CIRCLE_CENTER = (0.55, 0.20, 0.55)   # base 기준 원의 중심

## 2. 초기화

In [ ]:
import rclpy
from rclpy.node import Node
from rclpy.action import ActionClient
from rclpy.parameter import Parameter
from sensor_msgs.msg import JointState
from moveit_msgs.action import MoveGroup, ExecuteTrajectory
from moveit_msgs.srv import GetCartesianPath
from moveit_msgs.msg import RobotState
from visualization_msgs.msg import MarkerArray, Marker
from std_msgs.msg import ColorRGBA
from geometry_msgs.msg import Point

In [ ]:
try:
    rclpy.init()
except RuntimeError:
    pass  # 이미 초기화된 경우 무시

In [ ]:
node = Node(
    'franka_ex13_circle_demo',
    parameter_overrides=[Parameter('use_sim_time', value=True)],
)
move_client = ActionClient(node, MoveGroup, 'move_action')

joint_state = {'msg': None}
node.create_subscription(
    JointState, 'joint_states',
    lambda msg: joint_state.update(msg=msg), 10,
)
node.get_logger().info('=== franka_ex13 노트북 노드 생성 완료 ===')
marker_pub = node.create_publisher(MarkerArray, MARKER_TOPIC, 10)
cart_client = node.create_client(GetCartesianPath, 'compute_cartesian_path')

## 3. 서버 / 서비스 준비

In [ ]:
import time

def wait_for_ready(timeout_sec: float = 30.0):
    if not move_client.wait_for_server(timeout_sec=timeout_sec):
        raise RuntimeError('MoveGroup 액션 서버 연결 실패')
    if not cart_client.wait_for_service(timeout_sec=timeout_sec):
        raise RuntimeError('compute_cartesian_path 서비스 연결 실패')
    start = time.time()
    while joint_state['msg'] is None:
        rclpy.spin_once(node, timeout_sec=0.1)
        if time.time() - start > timeout_sec:
            raise RuntimeError('joint_states 수신 실패')
    node.get_logger().info('servers + /joint_states 준비됨')

wait_for_ready()

## 4. SRDF `ready` 자세

In [ ]:
from rclpy.parameter_client import AsyncParameterClient
import xml.etree.ElementTree as ET

def fetch_srdf_xml(timeout_sec: float = 10.0) -> str:
    client = AsyncParameterClient(node, 'move_group')
    if not client.wait_for_services(timeout_sec=timeout_sec):
        raise RuntimeError('move_group 파라미터 서비스 연결 실패')
    future = client.get_parameters(['robot_description_semantic'])
    rclpy.spin_until_future_complete(node, future, timeout_sec=timeout_sec)
    return future.result().values[0].string_value

def parse_named_pose(srdf_xml: str, name: str, group: str) -> dict:
    root = ET.fromstring(srdf_xml)
    for gs in root.findall('group_state'):
        if gs.attrib.get('group') == group and gs.attrib.get('name') == name:
            return {j.attrib['name']: float(j.attrib.get('value', '0'))
                    for j in gs.findall('joint')}
    raise RuntimeError(f'SRDF group_state "{name}" (group={group}) 없음')

def load_named_pose(name: str, timeout_sec: float = 10.0) -> dict:
    return parse_named_pose(fetch_srdf_xml(timeout_sec), name, PLANNING_GROUP)

ready_target = load_named_pose('ready')
node.get_logger().info(f'ready: {ready_target}')

## 5. Pose / MoveGroup / Execute 헬퍼

In [ ]:
import math
import tf_transformations
from geometry_msgs.msg import Pose, Point, Quaternion

def euler_to_quaternion(roll: float, pitch: float, yaw: float) -> Quaternion:
    q = tf_transformations.quaternion_from_euler(roll, pitch, yaw)
    return Quaternion(x=q[0], y=q[1], z=q[2], w=q[3])

def make_pose(x: float, y: float, z: float,
              roll: float = 0.0, pitch: float = 0.0, yaw: float = 0.0) -> Pose:
    pose = Pose()
    pose.position = Point(x=x, y=y, z=z)
    pose.orientation = euler_to_quaternion(roll, pitch, yaw)
    return pose

In [ ]:
from moveit_msgs.msg import (
    Constraints, JointConstraint,
    PositionConstraint, OrientationConstraint, BoundingVolume,
    MotionPlanRequest, PlanningOptions, MoveItErrorCodes,
)
from shape_msgs.msg import SolidPrimitive
from geometry_msgs.msg import Vector3

def make_joint_constraints(joint_values: dict, tol: float = 0.01) -> Constraints:
    c = Constraints()
    for jname, val in joint_values.items():
        c.joint_constraints.append(JointConstraint(
            joint_name=jname, position=val,
            tolerance_above=tol, tolerance_below=tol, weight=1.0,
        ))
    return c

def make_position_constraint(pose: Pose, tol: float = 0.01) -> PositionConstraint:
    pc = PositionConstraint()
    pc.header.frame_id = REFERENCE_FRAME
    pc.link_name = END_EFFECTOR_LINK
    pc.target_point_offset = Vector3(x=0.0, y=0.0, z=0.0)
    bv = BoundingVolume()
    sphere = SolidPrimitive()
    sphere.type = SolidPrimitive.SPHERE
    sphere.dimensions = [tol]
    bv.primitives.append(sphere)
    sp = Pose()
    sp.position = Point(x=pose.position.x, y=pose.position.y, z=pose.position.z)
    sp.orientation.w = 1.0
    bv.primitive_poses.append(sp)
    pc.constraint_region = bv
    pc.weight = 1.0
    return pc

def make_orientation_constraint(pose_or_quat, tol: float = 0.01) -> OrientationConstraint:
    oc = OrientationConstraint()
    oc.header.frame_id = REFERENCE_FRAME
    oc.link_name = END_EFFECTOR_LINK
    if hasattr(pose_or_quat, 'orientation'):
        oc.orientation = pose_or_quat.orientation
    else:
        oc.orientation = pose_or_quat
    oc.absolute_x_axis_tolerance = tol
    oc.absolute_y_axis_tolerance = tol
    oc.absolute_z_axis_tolerance = tol
    oc.weight = 1.0
    return oc

def make_plan_request(vel: float = 0.3, acc: float = 0.3,
                      attempts: int = 5, plan_time: float = 10.0,
                      planner_id: str = '') -> MotionPlanRequest:
    req = MotionPlanRequest()
    req.group_name = PLANNING_GROUP
    req.num_planning_attempts = attempts
    req.allowed_planning_time = plan_time
    req.max_velocity_scaling_factor = vel
    req.max_acceleration_scaling_factor = acc
    if planner_id:
        req.planner_id = planner_id
    return req

def send_move_goal(req: MotionPlanRequest, plan_only: bool = False):
    goal = MoveGroup.Goal()
    goal.request = req
    goal.planning_options = PlanningOptions(
        plan_only=plan_only, replan=not plan_only, replan_attempts=3 if not plan_only else 0)
    sf = move_client.send_goal_async(goal)
    rclpy.spin_until_future_complete(node, sf)
    handle = sf.result()
    if handle is None or not handle.accepted:
        return MoveItErrorCodes.PLANNING_FAILED, None
    rf = handle.get_result_async()
    rclpy.spin_until_future_complete(node, rf)
    res = rf.result().result
    return res.error_code.val, res.planned_trajectory

def go_to_joint_goal(joint_values: dict, vel: float = 0.3, acc: float = 0.3) -> bool:
    req = make_plan_request(vel, acc)
    req.goal_constraints.append(make_joint_constraints(joint_values))
    code_val, _ = send_move_goal(req, plan_only=False)
    ok = (code_val == MoveItErrorCodes.SUCCESS)
    if not ok:
        node.get_logger().error(f'joint goal 실패 error_code={code_val}')
    return ok

def go_to_pose_goal(pose: Pose, vel: float = 0.3, acc: float = 0.3) -> bool:
    req = make_plan_request(vel, acc)
    c = Constraints()
    c.position_constraints.append(make_position_constraint(pose))
    c.orientation_constraints.append(make_orientation_constraint(pose))
    req.goal_constraints.append(c)
    code_val, _ = send_move_goal(req, plan_only=False)
    ok = (code_val == MoveItErrorCodes.SUCCESS)
    if not ok:
        node.get_logger().error(f'pose goal 실패 error_code={code_val} (IK 해 없음 가능)')
    return ok

def plan_to_joint_goal(joint_values: dict, vel: float = 0.3, acc: float = 0.3,
                       planner_id: str = '', plan_time: float = 10.0):
    req = make_plan_request(vel, acc, plan_time=plan_time, planner_id=planner_id)
    req.goal_constraints.append(make_joint_constraints(joint_values))
    code_val, traj = send_move_goal(req, plan_only=True)
    return code_val == MoveItErrorCodes.SUCCESS, traj

def plan_to_pose_goal(pose: Pose, vel: float = 0.3, acc: float = 0.3,
                      planner_id: str = '', plan_time: float = 10.0):
    req = make_plan_request(vel, acc, plan_time=plan_time, planner_id=planner_id)
    c = Constraints()
    c.position_constraints.append(make_position_constraint(pose))
    c.orientation_constraints.append(make_orientation_constraint(pose))
    req.goal_constraints.append(c)
    code_val, traj = send_move_goal(req, plan_only=True)
    return code_val == MoveItErrorCodes.SUCCESS, traj

In [ ]:
from moveit_msgs.action import ExecuteTrajectory

execute_client = ActionClient(node, ExecuteTrajectory, 'execute_trajectory')
if not execute_client.wait_for_server(timeout_sec=15.0):
    raise RuntimeError('ExecuteTrajectory 액션 서버 연결 실패')

def execute_trajectory(trajectory) -> bool:
    g = ExecuteTrajectory.Goal()
    g.trajectory = trajectory
    sf = execute_client.send_goal_async(g)
    rclpy.spin_until_future_complete(node, sf)
    handle = sf.result()
    if handle is None or not handle.accepted:
        return False
    rf = handle.get_result_async()
    rclpy.spin_until_future_complete(node, rf)
    return rf.result().result.error_code.val == MoveItErrorCodes.SUCCESS

## 6. Cartesian path 헬퍼

`waypoints` 사이를 직선으로 잇고 `max_step` 마다 IK 를 풀어 trajectory 생성.

In [ ]:
def get_current_state() -> RobotState:
    rs = RobotState()
    if joint_state['msg'] is not None:
        rs.joint_state = joint_state['msg']
    return rs

def compute_cartesian_path(waypoints, max_step: float = 0.02,
                            avoid_collisions: bool = True,
                            vel: float = 0.2, acc: float = 0.2):
    req = GetCartesianPath.Request()
    req.header.frame_id = REFERENCE_FRAME
    req.group_name = PLANNING_GROUP
    req.link_name = END_EFFECTOR_LINK
    req.waypoints = list(waypoints)
    req.max_step = max_step
    req.avoid_collisions = avoid_collisions
    req.max_velocity_scaling_factor = vel
    req.max_acceleration_scaling_factor = acc
    req.start_state = get_current_state()
    fut = cart_client.call_async(req)
    rclpy.spin_until_future_complete(node, fut)
    res = fut.result()
    if res.error_code.val == MoveItErrorCodes.SUCCESS:
        node.get_logger().info(f'Cartesian 계획 성공 (달성률 {res.fraction*100:.1f}%)')
        return res.solution, res.fraction
    node.get_logger().error(f'Cartesian 계획 실패 (code={res.error_code.val})')
    return None, 0.0

## 7. 원 위 waypoints 생성 (YZ 평면)

회전축 = +X. 끝단 자세는 `pitch=+π/2` 로 고정 (그리퍼가 +X 방향을 향함).
각 θ 에서:
  pos = (cx, cy + r·sin(θ), cz + r·cos(θ))

In [ ]:
def generate_circle_yz(cx, cy, cz, radius, pts_per_loop, num_loops):
    waypoints = []
    orientation = euler_to_quaternion(0.0, math.pi / 2, 0.0)
    total = pts_per_loop * num_loops
    for i in range(total + 1):
        theta = 2.0 * math.pi * i / pts_per_loop
        wp = Pose()
        wp.position = Point(
            x=cx,
            y=cy + radius * math.sin(theta),
            z=cz + radius * math.cos(theta),
        )
        wp.orientation = orientation
        waypoints.append(wp)
    return waypoints

## 8. 마커 — 목표 원 + 시작점

In [ ]:
COLOR_TARGET = ColorRGBA(r=0.1, g=0.8, b=0.1, a=1.0)
COLOR_START  = ColorRGBA(r=1.0, g=0.0, b=0.0, a=1.0)
COLOR_ACTUAL = ColorRGBA(r=1.0, g=0.5, b=0.0, a=0.85)

def publish_target_circle(waypoints):
    stamp = node.get_clock().now().to_msg()
    line = Marker()
    line.header.frame_id = REFERENCE_FRAME
    line.header.stamp = stamp
    line.ns = 'target_circle'
    line.id = 0
    line.type = Marker.LINE_STRIP
    line.action = Marker.ADD
    line.scale.x = 0.005
    line.color = COLOR_TARGET
    line.pose.orientation.w = 1.0
    line.points = [Point(x=wp.position.x, y=wp.position.y, z=wp.position.z) for wp in waypoints]

    start = Marker()
    start.header.frame_id = REFERENCE_FRAME
    start.header.stamp = stamp
    start.ns = 'start'
    start.id = 0
    start.type = Marker.SPHERE
    start.action = Marker.ADD
    start.scale.x = start.scale.y = start.scale.z = 0.025
    start.color = COLOR_START
    start.pose = waypoints[0]

    ma = MarkerArray(markers=[line, start])
    marker_pub.publish(ma)

## 9. 시나리오 — 원 중심 → 시작점 → 카운트다운 → 실행

In [ ]:
# 9-1. ready 자세
go_to_joint_goal(ready_target, vel=0.4)
time.sleep(0.5)

# 9-2. 원 중심으로 이동 (단순 pose goal)
cx, cy, cz = CIRCLE_CENTER
center_pose = Pose()
center_pose.position = Point(x=cx, y=cy, z=cz)
center_pose.orientation = euler_to_quaternion(0.0, math.pi / 2, 0.0)
node.get_logger().info(f'--- Step 1: 원의 중심 ({cx}, {cy}, {cz}) 으로 이동 ---')
ok = go_to_pose_goal(center_pose, vel=0.3)
if not ok:
    raise RuntimeError('원의 중심 이동 실패 — center 좌표를 reach 안쪽으로 옮겨 보세요')
time.sleep(1.0)

# 9-3. waypoints 생성 + 마커
waypoints = generate_circle_yz(cx, cy, cz, RADIUS, PTS_PER_LOOP, NUM_LOOPS)
node.get_logger().info(f'YZ 원 waypoints {len(waypoints)} 개 (r={RADIUS*100:.0f}cm, {NUM_LOOPS}회)')
publish_target_circle(waypoints)

# 9-4. 시작점 (= waypoints[0]) 으로 이동
node.get_logger().info('--- Step 2: 원의 시작점으로 이동 ---')
ok = go_to_pose_goal(waypoints[0], vel=0.3)
if not ok:
    raise RuntimeError('시작점 이동 실패')
time.sleep(1.0)

# 9-5. 카운트다운
for i in [3, 2, 1]:
    node.get_logger().info(f'  >>> {i}')
    time.sleep(1.0)
node.get_logger().info('  >>> START!')

# 9-6. Cartesian path 계획
traj, fraction = compute_cartesian_path(waypoints, max_step=0.02, vel=0.15, acc=0.15)
if traj is None or fraction < 0.5:
    raise RuntimeError(f'경로 계획 실패 (fraction={fraction*100:.1f}%) — 중심 위치/반경 조정 필요')
node.get_logger().info(f'fraction = {fraction*100:.1f}% — 실행 시작')

# 9-7. 실행
ok = execute_trajectory(traj)
node.get_logger().info(f'원형 경로 실행 결과: {"성공" if ok else "실패"}')

## 10. 실제 경로 측정 + 시각화 (선택)

TF 로 끝단 위치를 샘플링해 LINE_STRIP 으로 그리는 간단 측정.
원이 도는 동안 별도 스레드에서 돌리는 게 정확하지만, 노트북에서는 사후 측정으로 충분.

In [ ]:
# 사후에 한 번 끝단 위치를 표시 (단순화 — 누적 추적은 ex14 참고)
import tf2_ros
buf = tf2_ros.Buffer()
listener = tf2_ros.TransformListener(buf, node)
for _ in range(20):
    rclpy.spin_once(node, timeout_sec=0.1)
try:
    tr = buf.lookup_transform(REFERENCE_FRAME, END_EFFECTOR_LINK, rclpy.time.Time())
    p = tr.transform.translation
    node.get_logger().info(f'현재 EE 위치: ({p.x:.3f}, {p.y:.3f}, {p.z:.3f})')
except Exception as e:
    node.get_logger().warn(f'TF lookup 실패: {e}')

## 11. ready 복귀

In [ ]:
go_to_joint_goal(ready_target, vel=0.4)
node.get_logger().info('=== franka_ex13 완료! ===')

## 12. 정리

In [ ]:
node.destroy_node()
try:
    rclpy.shutdown()
except Exception:
    pass